# 🥈 SILVER Notebook 2 of 3
### Rio Mart · 10-minute grocery delivery

**Rule of Silver: make it trustworthy.**
Fix the four problems Bronze exposed, and **derive** the columns the business
actually talks about — the ones no source system ever sends you.

| Derived column | Business meaning |
|---|---|
| `delivery_minutes` | how long the delivery really took |
| `is_late` | did we break the 10-minute promise? |
| `net_amount` | what we actually earned after discount |

```
  /Volumes  →  🥉 BRONZE  →  🥈 SILVER  →  🥇 GOLD
                            you are here
```

⚠️ Run `01_bronze_layer` first.

In [0]:
CATALOG = "workspace"
SCHEMA  = "rio_mart_quick_commerce"

spark.sql(f"USE {CATALOG}.{SCHEMA}")

from pyspark.sql import functions as F

## Step 1 — `silver_orders`

Four fixes and four derived columns, in one readable form

In [0]:
silver_orders = (
    spark.table("bronze_orders")
        .dropDuplicates(["order_id"])                                   # fix 1: kill the replays
        .filter(F.col("order_amount") > 0)                              # fix 2: drop junk / test orders
        .withColumn("order_status", F.upper(F.trim("order_status")))    # fix 3: DELIVERED / CANCELLED
        .withColumn("payment_mode", F.upper(F.trim("payment_mode")))    # fix 4: standardise payments
        .withColumn("payment_mode", F.when(F.col("payment_mode") == "CASH ON DELIVERY", "COD")
                                     .otherwise(F.col("payment_mode")))
        .withColumn("order_date", F.to_date("order_placed_ts"))
        .withColumn("order_hour", F.hour("order_placed_ts"))
        .withColumn(
            "delivery_minutes",
            F.round((F.unix_timestamp("delivered_ts") - F.unix_timestamp("order_placed_ts")) / 60, 1),
        )
        .withColumn("is_late", F.col("delivery_minutes") > F.col("promised_minutes"))
        .withColumn("net_amount", F.round(F.col("order_amount") - F.col("discount_amount"), 2))
        .select(
            "order_id", "store_id", "rider_id", "customer_id",
            "order_placed_ts", "delivered_ts", "order_date", "order_hour",
            "order_status", "payment_mode", "promised_minutes", "delivery_minutes", "is_late",
            "item_count", "order_amount", "discount_amount", "net_amount",
        )
)

silver_orders.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver_orders")
display(spark.table("silver_orders").limit(10))

order_id,store_id,rider_id,customer_id,order_placed_ts,delivered_ts,order_date,order_hour,order_status,payment_mode,promised_minutes,delivery_minutes,is_late,item_count,order_amount,discount_amount,net_amount
ORD000895,ST003,R0073,CUST03098,2026-07-27T14:15:54.000Z,2026-07-27T14:25:06.000Z,2026-07-27,14,DELIVERED,UPI,10,9.2,false,2,1052.4,0.0,1052.4
ORD004784,ST005,R0025,CUST00269,2026-08-16T12:31:53.000Z,2026-08-16T12:42:35.000Z,2026-08-16,12,DELIVERED,CARD,10,10.7,true,6,697.1,0.0,697.1
ORD000327,ST010,R0080,CUST02607,2026-07-24T13:28:44.000Z,2026-07-24T13:37:56.000Z,2026-07-24,13,DELIVERED,COD,10,9.2,false,6,1815.35,0.0,1815.35
ORD001537,ST007,R0057,CUST00791,2026-07-31T21:51:16.000Z,2026-07-31T22:00:58.000Z,2026-07-31,21,DELIVERED,UPI,10,9.7,false,2,130.5,0.0,130.5
ORD000751,ST003,R0073,CUST02893,2026-07-26T12:21:47.000Z,2026-07-26T12:42:29.000Z,2026-07-26,12,DELIVERED,UPI,10,20.7,true,6,1115.2,0.0,1115.2
ORD002404,ST006,R0076,CUST02496,2026-08-04T21:12:34.000Z,2026-08-04T21:21:04.000Z,2026-08-04,21,DELIVERED,CARD,10,8.5,false,4,489.65,73.45,416.2
ORD003296,ST004,R0044,CUST01990,2026-08-09T09:52:02.000Z,2026-08-09T09:59:32.000Z,2026-08-09,9,DELIVERED,WALLET,10,7.5,false,6,1936.5,0.0,1936.5
ORD000402,ST007,R0027,CUST01529,2026-07-25T22:21:58.000Z,2026-07-25T22:27:10.000Z,2026-07-25,22,DELIVERED,WALLET,10,5.2,false,4,1449.0,0.0,1449.0
ORD003655,ST007,R0087,CUST00941,2026-08-11T21:20:07.000Z,2026-08-11T21:28:19.000Z,2026-08-11,21,DELIVERED,CARD,10,8.2,false,1,250.0,0.0,250.0
ORD003116,ST002,R0012,CUST00350,2026-08-08T20:23:18.000Z,2026-08-08T20:38:30.000Z,2026-08-08,20,DELIVERED,UPI,10,15.2,true,4,600.0,0.0,600.0


> `delivery_minutes` is **null** for cancelled orders — there is no delivery time,
> so `is_late` is null too. That is deliberate, and it makes the Gold layer honest.

## Step 2 — `silver_stores`

One city was written five different ways. Fix it once and every
downstream table inherits the fix

In [0]:
silver_stores = (
    spark.table("bronze_dark_stores")
        .withColumn("city", F.initcap(F.trim("city")))
        .withColumn("area", F.trim("area"))
        .select("store_id", "store_name", "city", "area", "pincode", "opened_date")
)

silver_stores.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver_stores")
display(spark.table("silver_stores"))

store_id,store_name,city,area,pincode,opened_date
ST001,Rio Mart Koramangala,Bengaluru,Koramangala,560034,2024-03-11
ST002,Rio Mart Indiranagar,Bengaluru,Indiranagar,560038,2024-05-02
ST003,Rio Mart HSR Layout,Bengaluru,HSR Layout,560102,2024-07-19
ST004,Rio Mart Andheri West,Mumbai,Andheri West,400053,2024-04-08
ST005,Rio Mart Powai,Mumbai,Powai,400076,2024-09-23
ST006,Rio Mart Bandra,Mumbai,Bandra,400050,2025-01-14
ST007,Rio Mart Gachibowli,Hyderabad,Gachibowli,500032,2024-06-05
ST008,Rio Mart Madhapur,Hyderabad,Madhapur,500081,2025-02-27
ST009,Rio Mart Saket,Delhi,Saket,110017,2024-08-30
ST010,Rio Mart Gurgaon Sec 45,Delhi,Gurgaon Sector 45,122003,2025-03-18


## Step 3 — `silver_riders`

Attach the city to every rider by joining join two tables

In [0]:
silver_riders = (
    spark.table("bronze_riders").alias("r")
        .join(spark.table("silver_stores").alias("s"), "store_id", "left")
        .select(
            "r.rider_id", "r.rider_name", "r.store_id", "s.city",
            "r.vehicle_type", "r.joining_date", "r.is_active",
        )
)

silver_riders.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver_riders")
display(spark.table("silver_riders").limit(10))

rider_id,rider_name,store_id,city,vehicle_type,joining_date,is_active
R0001,Suresh Kumar,ST001,Bengaluru,EV Scooter,2025-10-16,true
R0002,Suresh Mehta,ST002,Bengaluru,EV Scooter,2024-12-12,true
R0003,Sunil Kumar,ST003,Bengaluru,Bike,2025-11-12,true
R0004,Jitendra Kumar,ST004,Mumbai,Bike,2024-08-27,true
R0005,Sunil Patil,ST005,Mumbai,Bike,2025-09-12,true
R0006,Arjun Gupta,ST006,Mumbai,EV Scooter,2025-04-17,true
R0007,Deepak Patil,ST007,Hyderabad,Bicycle,2025-07-14,true
R0008,Kiran Sharma,ST008,Hyderabad,Bike,2025-07-06,true
R0009,Ravi Rao,ST009,Delhi,Bike,2026-02-16,false
R0010,Kiran Sharma,ST010,Delhi,Bike,2024-09-18,true


## Step 4 — `silver_order_items`

Enrich every order line with its product details and its parent order's context

In [0]:
silver_order_items = (
    spark.table("bronze_order_items").alias("i")
        .join(spark.table("bronze_products").alias("p"), "product_id", "inner")
        .join(spark.table("silver_orders").alias("o"), "order_id", "inner")
        .withColumn("line_amount", F.round(F.col("quantity") * F.col("unit_price"), 2))
        .select(
            "i.order_item_id", "i.order_id", "o.order_date", "o.order_status", "o.store_id",
            "i.product_id", "p.product_name", "p.category", "p.brand",
            "i.quantity", "i.unit_price", "line_amount",
        )
)

silver_order_items.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver_order_items")
display(spark.table("silver_order_items").limit(10))

order_item_id,order_id,order_date,order_status,store_id,product_id,product_name,category,brand,quantity,unit_price,line_amount
OI0000001,ORD000001,2026-07-23,DELIVERED,ST008,P044,Head & Shoulders 340ml,Personal Care,Head & Shoulders,2,365.75,731.5
OI0000002,ORD000001,2026-07-23,DELIVERED,ST008,P052,Himalaya Baby Wipes 72pc,Baby Care,Himalaya,2,179.1,358.2
OI0000003,ORD000001,2026-07-23,DELIVERED,ST008,P026,Lays Classic Salted 52g,Snacks,Lays,1,19.0,19.0
OI0000004,ORD000002,2026-07-23,DELIVERED,ST007,P018,Coriander Bunch,Fruits & Vegetables,Farm Fresh,1,15.0,15.0
OI0000005,ORD000002,2026-07-23,DELIVERED,ST007,P049,Pampers Pants M 32pc,Baby Care,Pampers,2,599.0,1198.0
OI0000006,ORD000002,2026-07-23,DELIVERED,ST007,P054,Whiskas Tuna 480g,Pet Care,Whiskas,3,320.0,960.0
OI0000007,ORD000002,2026-07-23,DELIVERED,ST007,P027,Kurkure Masala Munch 90g,Snacks,Kurkure,1,20.0,20.0
OI0000008,ORD000002,2026-07-23,DELIVERED,ST007,P017,Green Capsicum 500g,Fruits & Vegetables,Farm Fresh,2,44.0,88.0
OI0000009,ORD000002,2026-07-23,DELIVERED,ST007,P058,Sumeru Green Peas 500g,Frozen,Sumeru,2,130.0,260.0
OI0000010,ORD000003,2026-07-23,DELIVERED,ST004,P048,Nivea Body Lotion 400ml,Personal Care,Nivea,2,425.0,850.0


## Step 5 — Did the cleaning actually work?

Run the same checks which was ran on Bronze

In [0]:
%sql
-- Seven spellings became two. Duplicates are gone.
SELECT order_status, count(*) AS orders, count(DISTINCT order_id) AS unique_orders
FROM silver_orders
GROUP BY order_status

order_status,orders,unique_orders
DELIVERED,5283,5283
CANCELLED,324,324


In [0]:
%sql
-- Five spellings of the cities became four real cities
SELECT city, count(*) AS stores
FROM silver_stores
GROUP BY city
ORDER BY city

city,stores
Bengaluru,3
Delhi,2
Hyderabad,2
Mumbai,3


## ✅ Silver is done

| Table | What changed |
|---|---|
| `silver_orders` | deduplicated, standardised, + 5 derived columns |
| `silver_stores` | city names conformed |
| `silver_riders` | city attached to each rider |
| `silver_order_items` | product and order context joined in |

The data is now trustworthy

➡️ **Next: open `03_gold_layer`.**